# Assignment 01 solutions

---

## Isomorphic Strings (LC 205)

See the assignment notebook for the full problem statement.

**Solution: technical note**

We scan `s` and `t` left to right in lock step. At each position `i` we have a pair `(s_char, t_char)` and need to enforce a one-to-one correspondence:

- if `s_char` has been seen before, it must map to the same `t_char` as last time;
- if `t_char` has been seen before, it must come from the same `s_char` as last time.

Since we can't use a dict or set, we track "characters already seen" with two parallel lists, `s_seen` and `t_seen`: `s_seen[j]` and `t_seen[j]` are a matched pair, and the shared index `j` is what keeps the mapping together. Checking "has this character been seen" means walking the relevant list with a loop -- that linear scan is the whole point of banning the hash-based shortcut.

We check *both* directions (`s -> t` and `t -> s`) because checking only one lets two different `s` characters map to the same `t` character undetected.

In [ ]:
def are_isomorphic(s: str, t: str) -> bool:
    """Given two strings, `s` and `t`, of the same length, determine whether
    the two strings are isomorphic. Two strings are isomorphic if we can
    replace each character in `s` with another character to obtain `t`,
    following these rules:

      1. Each character in `s` must always map to the same character in `t`.
      2. Different characters in `s` must map to different characters in `t`.
      3. A character is allowed to map to itself.
      4. The order of characters must remain the same.

    Uses only plain lists -- no dict/set -- per the assignment constraints.
    """

    MAX_LENGTH: int = 500

    # Return variable, first used to validate the inputs. s/t are annotated
    # str, but callers can still pass None/other types at runtime, so we
    # still guard for it.
    result: bool = (
        s is not None
        and t is not None
        and s.isalpha()
        and t.isalpha()
        and len(s) == len(t)
        and len(s) > 0
        and len(s) <= MAX_LENGTH
    )

    if result:
        # Characters from s and t already seen, scanning left to right.
        # s_seen[j] and t_seen[j] are always a matched pair.
        s_seen: list[str] = []
        t_seen: list[str] = []

        i = 0
        while i < len(s) and result:
            s_char = s[i]
            t_char = t[i]

            # Has s_char been seen before? If so, the t character it was
            # paired with must match today's t_char.
            s_char_seen_before = False
            j = 0
            while j < len(s_seen):
                if s_seen[j] == s_char:
                    s_char_seen_before = True
                    if t_seen[j] != t_char:
                        result = False
                j += 1

            # Same check in the other direction, to catch two different
            # s characters mapping to the same t character.
            t_char_seen_before = False
            k = 0
            while k < len(t_seen):
                if t_seen[k] == t_char:
                    t_char_seen_before = True
                    if s_seen[k] != s_char:
                        result = False
                k += 1

            # First time we've seen this pair -- remember it.
            if not s_char_seen_before and not t_char_seen_before:
                s_seen.append(s_char)
                t_seen.append(t_char)

            i += 1

    return result


In [ ]:
def _test_are_isomorphic():
    cases = [
        ("egg", "add", True),
        ("foo", "bar", False),
        ("paper", "title", True),
        ("badc", "baba", False),
        ("", "", False),          # too short per constraints
        ("ab", "aa", False),      # two different s chars mapping to same t char
        (None, "ab", False),
        ("ab1", "cd2", False),    # not alpha
    ]
    for s, t, expected in cases:
        # Some cases deliberately pass a non-str (e.g. None) to exercise
        # the required invalid-input handling; suppress ty's resulting
        # complaint about violating the str annotation.
        got = are_isomorphic(s, t)  # ty: ignore[invalid-argument-type]
        status = "OK" if got == expected else "FAIL"
        print(f"{status}: are_isomorphic({s!r}, {t!r}) = {got} (expected {expected})")


if __name__ == "__main__":
    _test_are_isomorphic()


---

## Longest Balanced Binary Subarray (LC 525)

See the assignment notebook for the full problem statement.

**Solution: technical note**

The stated constraint (`n <= 2,000`) is deliberately small enough that a brute-force approach is acceptable: for every starting index `i`, extend a subarray to the right one element at a time, keeping a running count of `0`s and `1`s seen so far, and whenever those counts are equal, check whether this subarray is the longest balanced one found yet.

This is O(n^2) -- for each of the `n` starting positions we may scan up to `n` elements -- deliberately, since the linear-time approach (prefix sums plus a hash map from running difference to first-seen index) needs the very hash-based structure this problem set is designed to avoid at this stage.

Invalid input (not a list, wrong element types, `None`, ...) returns `0` -- the same value used for "no balanced subarray exists" -- since the problem doesn't define a separate invalid-input signal.

In [ ]:
def contiguous_length(nums: list[int]) -> int:
    """Return the maximum length of a contiguous subarray of `nums` with an
    equal number of 0s and 1s, or 0 if no such subarray exists (including
    when `nums` is not a valid binary list -- the parameter is annotated
    list[int] for the intended/normal case, but we still validate at
    runtime since a caller can pass anything).

    Brute force: try every starting index, extend to the right, and track
    the best balanced length seen. O(n^2), acceptable per this problem's
    constraints (n <= 2,000).
    """

    valid: bool = isinstance(nums, list) and len(nums) >= 1
    if valid:
        for value in nums:
            if value != 0 and value != 1:
                valid = False

    best: int = 0

    if valid:
        n: int = len(nums)
        i = 0
        while i < n:
            zeros = 0
            ones = 0
            j = i
            while j < n:
                if nums[j] == 0:
                    zeros += 1
                else:
                    ones += 1

                if zeros == ones:
                    best = max(best, j - i + 1)

                j += 1
            i += 1

    return best


In [ ]:
def _test_contiguous_length():
    cases = [
        ([0, 1], 2),
        ([0, 1, 0], 2),
        ([0, 1, 1, 1, 1, 1, 0, 0, 0], 6),
        ([0, 0, 0], 0),
        ([1], 0),
        (None, 0),
        ([0, 1, 2], 0),     # invalid element
        ("01", 0),          # not a list
    ]
    for nums, expected in cases:
        # Some cases deliberately pass a non-list (e.g. None, a str) to
        # exercise the required invalid-input handling; suppress ty's
        # resulting complaint about violating the list[int] annotation.
        got = contiguous_length(nums)  # ty: ignore[invalid-argument-type]
        status = "OK" if got == expected else "FAIL"
        print(f"{status}: contiguous_length({nums!r}) = {got} (expected {expected})")


if __name__ == "__main__":
    _test_contiguous_length()


---

## Mirror Letters

Uses [`block_letters.py`](https://github.com/satyakisikdar/COMP-271-F26/blob/main/Week01/block_letters.py). See the assignment notebook for the full problem statement.

**Solution: technical note**

Two independent reversals are needed, and it's easy to accidentally do only one:

1. **Letter order** reverses -- the last letter in the word becomes the first.
2. **Each letter's own rows** reverse left-to-right, character by character.

Because `block_letters.m` and `block_letters.i` happen to be left-right symmetric, testing only with words built from those two letters can't catch a solution that forgot step 2 -- the output looks identical either way. Testing must include letters with asymmetric rows, like `block_letters.p` or `block_letters.s`.

Row reversal is written as an explicit loop rather than slicing (`row[::-1]`) or `reversed()`, per the assignment's constraint -- we walk the row from the last character to the first and build up the reversed string one character at a time.

Validation checks that `letters` is a list, that every letter has the same number of rows as the first letter, and that within each letter every row has the same width as that letter's first row. Any violation returns `None`.

In [ ]:
def mirror_letters(letters: list[list[str]]) -> list[list[str]] | None:
    """Given a word as a list of block letters (each a list of equal-width
    row strings), return the word as it would appear held up to a mirror:
    letter order reversed, and each letter's rows individually reversed
    left-to-right. Returns None if the input isn't a well-formed word.
    """

    valid: bool = isinstance(letters, list)

    height: int | None = None
    # letters[0] itself must be a list before we can measure its length --
    # otherwise a non-list first element (e.g. an int) crashes len() here
    # instead of falling through to the per-letter isinstance check below.
    if valid and len(letters) > 0 and isinstance(letters[0], list):
        height = len(letters[0])

    if valid:
        for letter in letters:
            if not isinstance(letter, list) or len(letter) != height:
                valid = False
            else:
                # Same guard one level down: a row must be a string before
                # len() can be called on it, or a letter like [[1, 2, 3]]
                # crashes here instead of being reported as invalid.
                first_row = letter[0] if len(letter) > 0 else ""
                if not isinstance(first_row, str):
                    valid = False
                else:
                    width = len(first_row)
                    for row in letter:
                        if not isinstance(row, str) or len(row) != width:
                            valid = False

    result: list[list[str]] | None = None

    if valid:
        result = []

        # Walk the letters back to front for the letter-order reversal.
        letter_index = len(letters) - 1
        while letter_index >= 0:
            letter = letters[letter_index]
            mirrored_letter = []

            for row in letter:
                # Manual left-to-right reversal of this row, character by
                # character -- no slicing, no reversed(), no .reverse().
                mirrored_row = ""
                pos = len(row) - 1
                while pos >= 0:
                    mirrored_row = mirrored_row + row[pos]
                    pos -= 1
                mirrored_letter.append(mirrored_row)

            result.append(mirrored_letter)
            letter_index -= 1

    return result


In [ ]:
import block_letters


def _test_mirror_letters():
    p_row0_reversed = block_letters.p[0][::-1]
    s_row0_reversed = block_letters.s[0][::-1]

    # "PS" mirrored: letter order flips to [S, P], each letter's rows flip.
    got = mirror_letters([block_letters.p, block_letters.s])
    expected_first_letter_row0 = s_row0_reversed
    expected_second_letter_row0 = p_row0_reversed
    ok = (
        got is not None
        and got[0][0] == expected_first_letter_row0
        and got[1][0] == expected_second_letter_row0
    )
    print("OK" if ok else "FAIL", ": mirror_letters([p, s]) row-0 check")

    # Empty word mirrors to an empty word.
    ok = mirror_letters([]) == []
    print("OK" if ok else "FAIL", ": mirror_letters([]) == []")

    # Mismatched letter heights -> None.
    broken_letter = block_letters.p[:-1]  # one row short
    ok = mirror_letters([block_letters.p, broken_letter]) is None
    print("OK" if ok else "FAIL", ": mismatched heights -> None")

    # Not a list -> None.
    # Deliberately wrong type, to exercise the runtime validation --
    # ty would otherwise flag this call against the list[list[str]]
    # annotation, so it's suppressed with ty's own ignore directive
    # (a different syntax from mypy's ignore comment).
    ok = mirror_letters("PS") is None  # ty: ignore[invalid-argument-type]
    print("OK" if ok else "FAIL", ": non-list input -> None")

    # A list whose *elements* aren't lists (e.g. ints) -- must return None,
    # not crash. Regression case: the first cut of this solution called
    # len(letters[0]) before confirming letters[0] was itself a list.
    ok = mirror_letters([5, 6]) is None  # ty: ignore[invalid-argument-type]
    print("OK" if ok else "FAIL", ": non-list elements -> None")

    # A letter (list) whose *rows* aren't strings -- same bug one level
    # down: must return None, not crash on len(letter[0]) or len(row).
    ok = mirror_letters([[1, 2, 3]]) is None  # ty: ignore[invalid-argument-type]
    print("OK" if ok else "FAIL", ": non-string rows -> None")


if __name__ == "__main__":
    _test_mirror_letters()
